# Setup

In [ ]:
import os
import scanpy as sc
import scipy
import squidpy as sq
import cell2location as c2l
import numpy as np
from gtfparse import read_gtf
import matplotlib
from pathlib import Path


sc.settings.verbosity = 3
sc.settings.set_figure_params(dpi=80, facecolor="white")
import pandas as pd
pd.options.display.max_seq_items = 100

In [2]:
import torch, os
print("CUDA visible:", os.environ.get("CUDA_VISIBLE_DEVICES"))
print("torch.cuda.is_available():", torch.cuda.is_available())
print("torch.version.cuda:", torch.version.cuda, "torch:", torch.__version__)

CUDA visible: 0,1
torch.cuda.is_available(): True
torch.version.cuda: 12.1 torch: 2.3.0+cu121


In [ ]:
results_folder = 'pdac_out_combined'

ref_run_name = f'{results_folder}/reference_signatures'
run_name = f'{results_folder}/cell2location_map'

os.makedirs(ref_run_name, exist_ok=True)
os.makedirs(run_name, exist_ok=True)

# Load Data

In [ ]:
seq_path = seq_path
adata_sn = sc.read(filename= seq_path/ 'hs_pdacatlas.h5ad')

In [ ]:
chen_adata = sc.read('GSE278694_raw.h5ad')
pei_adata = sc.read('Pei_raw.h5ad')
zhou_adata = sc.read('HTAN_raw.h5ad')

# Set up spatial data for cell2location

In [9]:
chen_adata.var["feature_name"] = chen_adata.var_names
pei_adata.var["feature_name"] = pei_adata.var_names
zhou_adata.var["feature_name"] = zhou_adata.var_names

In [13]:
chen_adata.var.set_index("gene_ids", drop=True, inplace=True)
pei_adata.var.set_index("gene_ids", drop=True, inplace=True)
zhou_adata.var.set_index("gene_ids", drop=True, inplace=True)

In [16]:
datasets = {
    "chen": chen_adata,   # may contain multiple modalities in .obs['modality']
    "pei":  pei_adata,
    "zhou": zhou_adata,
}

In [ ]:
def ensure_feature_name(adata, target_col="feature_name"):
    if target_col in adata.var:
        return target_col
    for cand in "gene_symbol","gene_name", "features"]:
        if cand in adata.var:
            adata.var[target_col] = adata.var[cand].astype(str)
            return target_col
    adata.var[target_col] = adata.var_names.astype(str)
    return target_col

In [ ]:
from scipy import sparse
def keep_mt_counts_then_drop_genes(adata, feature_col="feature_name", obsm_key="MT"):
    # keep raw counts for cell2location
    if "counts" not in adata.layers:
        adata.layers["counts"] = adata.X.copy()

    feature_col = ensure_feature_name(adata, feature_col)
    mt_mask = adata.var[feature_col].astype(str).str.upper().str.startswith("MT-").fillna(False).values

    X_mt = adata[:, mt_mask].X
    if sparse.issparse(X_mt): X_mt = X_mt.toarray()
    adata.obsm[obsm_key] = X_mt  

    mt_counts = X_mt.sum(1)
    tot = adata.X.sum(1).A1 if sparse.issparse(adata.X) else adata.X.sum(1)
    adata.obs["MT_counts"] = mt_counts
    adata.obs["MT_frac"]   = np.divide(mt_counts, tot, out=np.zeros_like(mt_counts, float), where=tot>0)

    adata._inplace_subset_var(~mt_mask)
    return adata

def _yesno_to_bool(s):
    return pd.Series(s).astype(str).str.strip().str.lower().map({"yes": True, "no": False}).fillna(False)

def add_chen_tech(adata, cyt_col="cytassist", ffpe_col="visium_ffpe"):
    cyt  = _yesno_to_bool(adata.obs[cyt_col])  if cyt_col  in adata.obs else pd.Series(False, index=adata.obs_names)
    ffpe = _yesno_to_bool(adata.obs[ffpe_col]) if ffpe_col in adata.obs else pd.Series(False, index=adata.obs_names)
    tech = np.where(ffpe & cyt, "Visium_FFPE_CytAssist",
           np.where(ffpe,       "Visium_FFPE",
           np.where(cyt,        "Visium_CytAssist", "Visium")))
    adata.obs["tech"] = pd.Categorical(tech)
    return adata

def ensure_tech(adata, default="Visium"):
    if "tech" not in adata.obs:
        adata.obs["tech"] = pd.Categorical([default] * adata.n_obs)
    return adata

def pick_slide_id(adata):
    for c in ["library_id","sample_id"]:
        if c in adata.obs: return c
    adata.obs["synthetic_slide"] = "slide0"
    return "synthetic_slide"

In [ ]:
for name, ad in datasets.items():
    ad.obs["dataset"] = name
    keep_mt_counts_then_drop_genes(ad)

    if name == "chen":
        add_chen_tech(ad)                  
    else:
        ensure_tech(ad, default="Visium") 

    slide_col = pick_slide_id(ad)
    ad.obs["batch_c2l"] = (
        ad.obs["dataset"].astype(str) + "|" +
        ad.obs["tech"].astype(str) + "|" +
        ad.obs[slide_col].astype(str)
    )

# Process single cell for cell2location

In [35]:
adata_sn.var["feature_name"] = adata_sn.var_names
adata_sn.var["gene_ids"] = adata_sn.var['feature_name'].apply(map_gene_to_id)

In [ ]:
adata_sn.var.set_index("gene_ids", drop=True, inplace=True)
adata_sn.var.head()

In [ ]:
print("Max value in X:", adata_sn.X.max())
print("Min value in X:", adata_sn.X.min())

# Concatenate spatial

In [ ]:
adata_st = sc.concat(
    [chen_adata, pei_adata, zhou_adata],
    label="batch_from_concat",
    keys=["chen", "pei", "zhou"],
    join="inner", 
    axis=0
)

In [46]:
shared_features = [
    feature for feature in adata_st.var_names if feature in adata_sn.var_names
]

In [54]:
adata_st.var.index.is_unique

True

In [55]:
len(shared_features)

16102

In [56]:
from anndata import AnnData
adata_sn = adata_sn[:, shared_features].copy()
adata_st = adata_st[:, shared_features].copy()

In [57]:
adata_sn.obs['source_patient_id'].unique()

['HTA12_6', 'HTA12_7', 'HTA12_8', 'HTA12_9', 'HTA12_10', ..., 'donor1', 'donor9', 'donor4', 'donor8', 'donor5']
Length: 94
Categories (94, object): ['1', '2', '3', '4', ..., 'donor5', 'donor6', 'donor8', 'donor9']

In [58]:
adata_st.obs['library_id'].unique()

['PA85', 'PA05-1', 'PA24', 'PA72', 'PA04-1', ..., 'HTA12_264_2', 'HTA12_265_1', 'HTA12_267_1', 'HTA12_268_2', 'HTA12_273_2']
Length: 57
Categories (57, object): ['HTA12_22_4', 'HTA12_23_3', 'HTA12_24_5', 'HTA12_24_6', ..., 'Pt-10A', 'Pt-11A', 'Pt-12A', 'Pt-13A']

# Fitting the reference model

In [ ]:
selected = c2l.utils.filtering.filter_genes(
    adata_sn, cell_count_cutoff=10, cell_percentage_cutoff2=0.03, nonz_mean_cutoff=1.12
)

In [60]:
len(selected)

10983

In [64]:
adata_sn = adata_sn[:, selected].copy()
adata_st = adata_st[:, selected].copy()

In [65]:
adata_sn.write(results_folder + "/filtered_adata_sc.h5ad")
adata_st.write(results_folder + "/filtered_adata_st.h5ad")

... storing 'dataset' as categorical
... storing 'batch_c2l' as categorical


# Merge annotations in single cell atlas

In [ ]:
map_dict = {
    # CD8
    'CD8 Teff':'CD8 Teff','CD8 Tem':'CD8 Teff','CD8 Tcm':'CD8 Teff',
    'CD8 Tex prog':'CD8 Tex','CD8 Tex int':'CD8 Tex','CD8 Tex term':'CD8 Tex',
    # CD4
    'CD4 Teff':'CD4 conventional','CD4 Tem':'CD4 conventional','CD4 Tcm':'CD4 conventional',
    'CD4 Treg':'CD4 Treg','CD4 Tfh':'CD4 Tfh',
    #  NK / γδ
    'NK cells':'NK cells','NK-like T cells':'NK cells','gd T cells':'gd T cells',
    # B lineage
    'B cells (naive)':'B cells','B cells (memory)':'B cells','Plasma cells':'Plasma cells',
    # Monocyte / DC
    'Monocytes (classical)':'Monocytes','Monocytes (non-classical)':'Monocytes',
    'DC1':'Dendritic cells','DC2':'Dendritic cells','pDC':'Dendritic cells','mregDC':'Dendritic cells',
    # Macrophage subclasses 
    'Macrophages (SPP1+MARCO+)':'Macrophages SPP1','Macrophages (IL1B+)':'Macrophages IL1B',
    'Macrophages (FOLR2+)':'Macrophages FOLR2','Macrophages (prolif.)':'Macrophages proliferating',
    # Neut/Mast
    'Neutrophils':'Neutrophils','Mast cells':'Mast cells',
    # Endothelial (collapse)
    'Endothelial cells (venous)':'Endothelial cells','Endothelial cells (capillary)':'Endothelial cells',
    'Endothelial cells (angiogenic)':'Endothelial cells','Endothelial cells (arterial)':'Endothelial cells',
    'Endothelial cells (lymphatic)':'Endothelial cells',
    # Mural
    'Pericytes':'Stromal cells','Smooth muscle cells':'Stromal cells','Perivascular fibroblasts':'Stromal cells',
    # Fibroblast (stick to iCAF/myoCAF)
    'iCAF':'iCAF','MyoCAF':'MyoCAF','ApCAF':'iCAF','Fibroblasts':'iCAF',
    # Epithelial
    'Cancer cells (classical)':'Cancer cells','Cancer cells (basal-like)':'Cancer cells','Cancer cells (exocrine-like)':'Cancer cells',
    'Acinar cells':'Acinar cells','Ductal cells':'Ductal cells','Islet cells':'Islet cells','Schwann cells':'Schwann cells',
    # ADM
    'ADM':'Ductal cells'   
}

import re

def norm_str(s: pd.Series) -> pd.Series:
    x = s.astype(str).str.normalize("NFKC").str.strip()
    x = x.str.replace(r'\s+', ' ', regex=True)                   
    x = x.str.replace(r'[\u2010-\u2015\u2212]', '-', regex=True) 
    return x

adata_sn.obs['anno_sub_norm'] = norm_str(adata_sn.obs['anno_sub'])

map_norm = { norm_str(pd.Series([k]))[0]: (norm_str(pd.Series([v]))[0] if v is not None else None)
             for k, v in map_dict.items() }

adata_sn.obs['anno_broad'] = adata_sn.obs['anno_sub_norm'].map(map_norm)

na_mask = adata_sn.obs['anno_broad'].isna()
print("Remaining NAs:", int(na_mask.sum()))
if na_mask.any():
    print(adata_sn.obs.loc[na_mask, 'anno_sub'].value_counts().head(10))

adata_sn.obs['anno_broad'] = pd.Categorical(adata_sn.obs['anno_broad'])

adata_sn.obs['anno_broad'] = adata_sn.obs['anno_sub'].map(map_dict)
adata_sn.obs['anno_broad'] = adata_sn.obs['anno_broad'].astype('category')

Remaining NAs: 0


In [ ]:
import pandas as pd
if 'treatment' not in adata_sn.obs:
    adata_sn.obs['treatment'] = pd.Series(['unknown'] * adata_sn.n_obs, index=adata_sn.obs_names)

t = adata_sn.obs['treatment']

if pd.api.types.is_categorical_dtype(t):
    if 'unknown' not in t.cat.categories:
        t = t.cat.add_categories(['unknown'])
    t = t.fillna('unknown')
else:
    t = t.astype(object).where(~pd.isna(t), 'unknown')
    t = t.replace(['', 'nan', 'NaN', 'None', None], 'unknown')
    t = t.astype('category')

adata_sn.obs['treatment'] = t
print(adata_sn.obs['treatment'].value_counts(dropna=False))

In [72]:
c2l.models.RegressionModel.setup_anndata(
    adata=adata_sn,
    batch_key="source",
    labels_key="anno_broad",
    categorical_covariate_keys=["treatment"],
    layer="counts",
)

In [ ]:
model = c2l.models.RegressionModel(adata_sn)
# default, try on GPU:
model.train(max_epochs=250, batch_size=2500, train_size=1, lr=0.002, accelerator="gpu")

In [ ]:
model.plot_history(10)

In [ ]:
model.export_posterior(
    adata_sn,
    sample_kwargs={"num_samples": 1000, "batch_size": 2500, "accelerator":"cpu"},
)

In [ ]:
model.plot_QC()

In [ ]:
# export estimated expression in each cluster
if "means_per_cluster_mu_fg" in adata_sn.varm.keys():
    inf_aver = adata_sn.varm["means_per_cluster_mu_fg"][
        [f"means_per_cluster_mu_fg_{i}" for i in adata_sn.uns["mod"]["factor_names"]]
    ].copy()
else:
    inf_aver = adata_sn.var[
        [f"means_per_cluster_mu_fg_{i}" for i in adata_sn.uns["mod"]["factor_names"]]
    ].copy()

inf_aver.columns = adata_sn.uns["mod"]["factor_names"]
inf_aver.head()

In [78]:
inf_aver.to_csv(results_folder + "/reference_signatures" + "/inf_aver.csv")
inf_aver.shape

(10984, 26)

# Cell2location cell type mapping in spatial

In [4]:
## If only reading saved cell type signatures
results_folder = Path(results_folder)
inf_aver = pd.read_csv(results_folder / "reference_signatures" / "inf_aver.csv", index_col=0)
adata_st = sc.read(results_folder / "filtered_adata_st.h5ad")
adata_sn = sc.read(results_folder / "filtered_adata_sc.h5ad")
inf_aver.shape

(10984, 26)

In [ ]:
c2l.models.Cell2location.setup_anndata(
    adata=adata_st,
    batch_key="library_id")

model = c2l.models.Cell2location(
    adata_st,
    detection_alpha=200,
    cell_state_df=inf_aver,
    N_cells_per_location=10)

model.view_anndata_setup()


In [ ]:
use_gpu = True
#torch.set_float32_matmul_precision('high')
model.train(batch_size=10000, 
            train_size=1, 
            max_epochs=2000,
            accelerator='gpu')
# plot training history
model.plot_history()

In [ ]:
adata_st = model.export_posterior(
    adata_st,
    sample_kwargs={
        "num_samples": 200,
        "batch_size": 1024 # low enough to fit on gpu
    },
)

adata_file = f"analysis_combined/resulting_adata_st.h5ad"
adata_st.write(adata_file)
adata_file

In [ ]:
from pathlib import Path

out_dir = Path("analysis_combined")
out_dir.mkdir(parents=True, exist_ok=True)   # make the folder(s) if missing

adata_file = out_dir / "resulting_adata_st.h5ad"
adata_st.write(adata_file)
print(f"saved to: {adata_file.resolve()}")

In [ ]:
adata_file = f"analysis_combined/resulting_adata_st.h5ad"
adata_st = sc.read_h5ad(adata_file)

In [ ]:
model.plot_QC()
adata_st

In [ ]:
adata_st.obsm["q05_cell_abundance_w_sf"]

In [22]:
adata_st.obs[adata_st.uns["mod"]["factor_names"]] = adata_st.obsm[
    "q05_cell_abundance_w_sf"
]

In [37]:
adata_st.write(adata_file)
print(f"saved to: {adata_file.resolve()}")

saved to: /sc/arion/projects/Perturb-map/BhavyaSingh/integration/analysis_combined/resulting_adata_st.h5ad
